# 생활권 군집화 이해하기 — DBSCAN, HDBSCAN, 그리고 우리 설계

이 노트북은 세 가지를 하려고 만들었습니다.

1. scikit-learn 공식 예제로 두 군집 방법(DBSCAN, HDBSCAN)이 뭔지 익히기
2. 우리 데이터에서 둘을 직접 돌려보고 비교하기
3. 우리 설계가 왜 지금 모양인지, 그리고 아직 열려 있는 고민이 뭔지 정리하기

어느 한쪽이 정답이라고 주장하는 문서가 아닙니다. 둘 다 쓸 수 있는 방법이고, 둘 다 이슈가 있습니다.
그 이슈가 뭔지, 우리가 어디까지 확인했고 뭘 아직 모르는지를 같이 보는 게 목적입니다.

사용법: 위 메뉴에서 런타임 → 모두 실행. 코드는 대부분 접혀 있어서 그림과 글만 보면 됩니다.
각 셀 왼쪽의 실행 버튼을 눌러 하나씩 따라가도 됩니다.

## 미리 알아둘 용어 5개

| 용어 | 뜻 |
|---|---|
| 거점 | 반복해서 가는 곳 (집, 시장, 병원). 좌표 점들이 뭉친 곳에서 발견됨 |
| eps | "몇 미터 안에 모인 방문을 같은 장소로 볼 것인가"라는 반경. DBSCAN에서 사람이 정하는 값 |
| 생활권 | 각 거점에 원을 씌우고(반경 500m~2km) 전부 합친 영역. 판정은 이 합친 영역 기준 |
| 노이즈 | 어느 거점에도 못 묶여서 제외된 방문 (한 번 가본 결혼식장 같은 것) |
| 산포 | 같은 장소를 가도 주차 위치나 GPS 오차 때문에 좌표가 흩어지는 정도. 보통 수십~수백 m |


In [ ]:
#@title 준비 — 데이터 다운로드와 한글 폰트 (실행만 하면 됩니다)
import sys, os, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("seniorcareservice"):
        !git clone --branch claude/gaip-dashboard-refine --depth 1 -q https://github.com/summit1123/seniorcareservice.git
    ROOT = "seniorcareservice"
    !apt-get -qq -y install fonts-nanum > /dev/null 2>&1
    import matplotlib.font_manager as fm
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    FONT = "NanumGothic"
else:
    ROOT = ".."
    FONT = "Malgun Gothic"
sys.path.insert(0, ROOT)

import matplotlib.pyplot as plt
plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = False
print("준비 완료")

---

# 1부. 공식 자료로 배우기

## 1-1. DBSCAN이 뭔가

한 문장으로: 점들이 밀집한 곳을 찾아 묶고, 어디에도 못 낀 점은 노이즈로 분류하는 방법입니다.
묶는 기준은 두 개뿐입니다. "얼마나 가까우면 이웃인가(eps)"와 "이웃이 몇 개 모여야 묶음으로 인정하나(min_samples)".

아래는 scikit-learn 공식 예제를 그대로 실행한 것입니다
(원본: https://scikit-learn.org/stable/auto_examples/cluster/plot_dbscan.html , 코드는 접어뒀고 펼치면 원문 그대로입니다).
가상의 점 750개를 세 무리로 뿌려놓고 DBSCAN이 그걸 찾게 한 실험입니다.

In [ ]:
#@title 공식 DBSCAN 예제 (원본 코드 — 펼치면 볼 수 있습니다)
# 출처: scikit-learn "Demo of DBSCAN clustering algorithm"
# Authors: The scikit-learn developers | SPDX-License-Identifier: BSD-3-Clause
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

centers = [[1, 1], [-1, -1], [1, -1]]
X, labels_true = make_blobs(
    n_samples=750, centers=centers, cluster_std=0.4, random_state=0
)

X = StandardScaler().fit_transform(X)

import numpy as np
from sklearn import metrics
from sklearn.cluster import DBSCAN

db = DBSCAN(eps=0.3, min_samples=10).fit(X)
labels = db.labels_

n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
n_noise_ = list(labels).count(-1)

print("Estimated number of clusters: %d" % n_clusters_)
print("Estimated number of noise points: %d" % n_noise_)
print(f"Silhouette Coefficient: {metrics.silhouette_score(X, labels):.3f}")

unique_labels = set(labels)
core_samples_mask = np.zeros_like(labels, dtype=bool)
core_samples_mask[db.core_sample_indices_] = True

colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]
for k, col in zip(unique_labels, colors):
    if k == -1:
        col = [0, 0, 0, 1]

    class_member_mask = labels == k

    xy = X[class_member_mask & core_samples_mask]
    plt.plot(xy[:, 0], xy[:, 1], "o", markerfacecolor=tuple(col),
             markeredgecolor="k", markersize=14)

    xy = X[class_member_mask & ~core_samples_mask]
    plt.plot(xy[:, 0], xy[:, 1], "o", markerfacecolor=tuple(col),
             markeredgecolor="k", markersize=6)

plt.title(f"Estimated number of clusters: {n_clusters_}")
plt.show()

위 그림을 읽는 법. 색깔 = DBSCAN이 찾은 무리(3개, 정답과 일치), 검은 점 = 노이즈로 분류된 점 18개.
큰 점은 무리의 중심부(core), 작은 점은 가장자리라는 뜻인데 지금은 몰라도 됩니다.

출력에 나온 Silhouette Coefficient는 "정답을 모를 때 군집이 잘 됐는지 재는 점수"입니다.
나중에 실데이터(정답 라벨이 없는)에서 결과를 검증할 때 이런 지표를 쓰게 되므로 이름만 기억해 두면 됩니다.

## 1-2. HDBSCAN이 뭔가

DBSCAN의 eps(반경)를 사람이 정하는 대신, 반경을 0부터 계속 키워가며 점들이 뭉치는 과정 전체를 지켜보고,
"넓은 반경 구간에서 오래 따로 유지된 묶음"을 군집으로 뽑는 방법입니다. 반경을 데이터가 정하게 한 셈입니다.

공식 예제(https://scikit-learn.org/stable/auto_examples/cluster/plot_hdbscan.html)가 세 가지 실험으로
이 차이를 보여줍니다. 하나씩 실행하며 보겠습니다.

In [ ]:
#@title 공식 HDBSCAN 예제 1 — 같은 데이터를 늘렸다 줄였다 하면? (원본 코드)
# 출처: scikit-learn "Demo of HDBSCAN clustering algorithm"
# Authors: The scikit-learn developers | SPDX-License-Identifier: BSD-3-Clause
from sklearn.cluster import HDBSCAN
from sklearn.datasets import make_blobs


def plot(X, labels, probabilities=None, parameters=None, ground_truth=False, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 4))
    labels = labels if labels is not None else np.ones(X.shape[0])
    probabilities = probabilities if probabilities is not None else np.ones(X.shape[0])
    unique_labels = set(labels)
    colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]
    proba_map = {idx: probabilities[idx] for idx in range(len(labels))}
    for k, col in zip(unique_labels, colors):
        if k == -1:
            col = [0, 0, 0, 1]

        class_index = (labels == k).nonzero()[0]
        for ci in class_index:
            ax.plot(
                X[ci, 0],
                X[ci, 1],
                "x" if k == -1 else "o",
                markerfacecolor=tuple(col),
                markeredgecolor="k",
                markersize=4 if k == -1 else 1 + 5 * proba_map[ci],
            )
    n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
    preamble = "True" if ground_truth else "Estimated"
    title = f"{preamble} number of clusters: {n_clusters_}"
    if parameters is not None:
        parameters_str = ", ".join(f"{k}={v}" for k, v in parameters.items())
        title += f" | {parameters_str}"
    ax.set_title(title)
    plt.tight_layout()


centers = [[1, 1], [-1, -1], [1.5, -1.5]]
X, labels_true = make_blobs(
    n_samples=750, centers=centers, cluster_std=[0.4, 0.1, 0.75], random_state=0
)
fig, axes = plt.subplots(3, 1, figsize=(10, 12))
dbs = DBSCAN(eps=0.3)
for idx, scale in enumerate([1, 0.5, 3]):
    dbs.fit(X * scale)
    plot(X * scale, dbs.labels_, parameters={"scale": scale, "eps": 0.3}, ax=axes[idx])

fig, axes = plt.subplots(3, 1, figsize=(10, 12))
hdb = HDBSCAN(copy=True)
for idx, scale in enumerate([1, 0.5, 3]):
    hdb.fit(X * scale)
    plot(X * scale, hdb.labels_, hdb.probabilities_, ax=axes[idx],
         parameters={"scale": scale})

실험 1이 보여주는 것: 같은 데이터를 0.5배로 줄이거나 3배로 늘리기만 해도, 같은 eps=0.3을 쓴
DBSCAN 결과가 완전히 달라집니다(위 3장). HDBSCAN은 축척과 무관하게 같은 답을 냅니다(아래 3장).

eps라는 값이 "데이터의 단위와 축척에 묶여 있는" 값이라는 뜻입니다. 우리가 좌표를 도(degree) 단위가 아니라
미터로 통일해서 쓰는 이유가 이것과 닿아 있습니다. 미터로 고정하면 축척 문제는 사라지지만,
"몇 미터가 적절한가"라는 질문은 남습니다. 그 질문이 이 노트북 후반부의 주제입니다.

In [ ]:
#@title 공식 HDBSCAN 예제 2 — 빽빽한 무리와 성긴 무리가 섞여 있으면? (원본 코드)
centers = [[-0.85, -0.85], [-0.85, 0.85], [3, 3], [3, -3]]
X, labels_true = make_blobs(
    n_samples=750, centers=centers, cluster_std=[0.2, 0.35, 1.35, 1.35], random_state=0
)
plot(X, labels=labels_true, ground_truth=True)

fig, axes = plt.subplots(2, 1, figsize=(10, 8))
params = {"eps": 0.7}
dbs = DBSCAN(**params).fit(X)
plot(X, dbs.labels_, parameters=params, ax=axes[0])
params = {"eps": 0.3}
dbs = DBSCAN(**params).fit(X)
plot(X, dbs.labels_, parameters=params, ax=axes[1])

hdb = HDBSCAN(copy=True).fit(X)
plot(X, hdb.labels_, hdb.probabilities_)

실험 2가 보여주는 것: 촘촘한 무리 2개와 넓게 퍼진 무리 2개가 한 판에 있으면,
eps를 크게(0.7) 잡으면 촘촘한 두 무리가 하나로 합쳐지고, 작게(0.3) 잡으면 퍼진 무리가 조각납니다.
어떤 eps를 골라도 넷을 동시에 못 잡습니다. HDBSCAN은 무리마다 다른 반경을 적용해 넷 다 찾습니다(마지막 그림).

이게 HDBSCAN이 만들어진 이유입니다. 다만 전제가 있습니다. "밀도가 다른 무리들이 한 판에 섞여 있을 때"의
이야기라는 것. 우리는 고객 한 명씩 따로 군집화를 돌리기 때문에 이 전제가 약해지고,
그래서 2부에서 보겠지만 우리 데이터에서는 둘의 결과가 거의 같게 나옵니다.

In [ ]:
#@title 공식 HDBSCAN 예제 3 — HDBSCAN의 설정을 바꾸면? (원본 코드)
PARAM = ({"min_cluster_size": 5}, {"min_cluster_size": 3}, {"min_cluster_size": 25})
fig, axes = plt.subplots(3, 1, figsize=(10, 12))
for i, param in enumerate(PARAM):
    hdb = HDBSCAN(copy=True, **param).fit(X)
    labels = hdb.labels_

    plot(X, labels, hdb.probabilities_, param, ax=axes[i])

PARAM = (
    {"min_cluster_size": 20, "min_samples": 5},
    {"min_cluster_size": 20, "min_samples": 3},
    {"min_cluster_size": 20, "min_samples": 25},
)
fig, axes = plt.subplots(3, 1, figsize=(10, 12))
for i, param in enumerate(PARAM):
    hdb = HDBSCAN(copy=True, **param).fit(X)
    labels = hdb.labels_

    plot(X, labels, hdb.probabilities_, param, ax=axes[i])

실험 3이 보여주는 것: HDBSCAN도 설정 두 개(min_cluster_size, min_samples)에 따라
결과가 꽤 달라집니다. 공식 문서 스스로 이 민감성을 다루고 있습니다.

여기서 기억할 것 하나. HDBSCAN이 eps를 없앤 건 맞지만, "얼마나 모여야 묶음인가"라는 질문 자체가
사라진 건 아닙니다. 질문의 단위가 "몇 미터"에서 "몇 개"로 바뀐 것에 가깝습니다.

## 1-3. 방금 본 것을 수식으로 정리

"이웃"이라는 말부터. 이웃 = 어떤 점 주변, 일정 거리 안에 있는 다른 점들.
두 방법의 차이는 그 "일정 거리"를 누가 정하느냐입니다.

| | DBSCAN | HDBSCAN |
|---|---|---|
| 이웃의 반경 | 사람이 정함: $\varepsilon$ | 데이터가 정함: 점마다 핵심거리 $\mathrm{core}_k(p)$ 계산 |
| 이웃 정의 | $N(p) = \{\, q : d(p,q) \le \varepsilon \,\}$ | 고정 반경 없음. 아래 연결 반경으로 대체 |
| 묶음 판정 | $N(p)$가 기준 이상이면 $p$에서 이웃을 따라 연결 | 두 점의 연결 반경 $d_{mr}(p,q)=\max(\mathrm{core}_k(p),\,\mathrm{core}_k(q),\,d(p,q))$ 로 뭉침 과정 전체를 기록하고, 오래 유지된 묶음을 선택 |

기호 읽는 법: $p, q$ = 점 하나씩. $d(p,q)$ = 두 점 사이 거리. $\varepsilon$ = eps.
$\mathrm{core}_k(p)$ = $p$에서 $k$번째로 가까운 점까지의 거리.

핵심거리를 숫자로 감 잡기: 점 A에서 가장 가까운 점이 100m, 그다음이 500m에 있으면
$\mathrm{core}_2(A) = 500$m입니다. "A 주변은 500m는 나가야 점 2개가 모이는 한산한 동네"라는 뜻이고,
그래서 A는 바로 옆 100m 점과도 반경이 500m로 커져야 연결됩니다. 한산한 곳의 우연한 몰림을
걸러내는 장치인데, 뒤집으면 점이 적을 때 판단이 흔들리는 원인이 되기도 합니다.

HDBSCAN 전체를 순서로 풀면 세 단계입니다.

1. 방문점마다 핵심거리를 잰다 — min_samples번째로 가까운 방문점까지의 거리. "내 주변이 얼마나 한산한가"의 점수.
2. 반경을 0부터 키워가며, 연결 반경이 닿는 점끼리 묶이는 과정 전체를 기록한다.
3. 그 기록에서, 넓은 반경 구간 동안 오래 따로 유지된 묶음을 군집으로 뽑는다.
   이때 방문점이 min_cluster_size개 미만인 묶음은 버린다.

여기서 "개수"는 전부 **방문 기록의 개수**입니다. min_cluster_size=5는 "방문 5개 미만인 뭉치는
거점이 못 된다", min_samples=3은 "가까운 방문이 3개는 있어야 밀집으로 본다"는 뜻입니다.
기준선 2개월에 방문이 60개쯤인 우리 데이터로 예를 들면, 2주에 두 번 간 병원(방문 4개)은
min_cluster_size=5 설정에서는 거점이 못 됩니다 — 그래서 "왜 5냐, 4는 안 되냐"는 질문이
eps의 "왜 260이냐"와 같은 종류의 질문으로 남습니다.

파라미터 전체 목록:

| 파라미터 | 어느 쪽 | 뜻 | 우리 값 | 값의 출처 |
|---|---|---|---|---|
| $\varepsilon$ (eps) | DBSCAN | 한 장소로 묶는 반경 | 도심 260m (지역 3단: 260/520/1,100) | 문헌 대역, 4부에서 정리 |
| min_days | DBSCAN 우리 변형 | 거점 인정에 필요한 서로 다른 방문일 | 3일 | 상품 규칙 + 선례 논문 (4부) |
| min_cluster_size | HDBSCAN | 묶음으로 인정하는 최소 점 개수 | 5로 실험 중 | 정해진 근거 없음, 민감 |
| min_samples ($k$) | HDBSCAN | 핵심거리의 $k$. 밀집 판단의 보수성 | 3으로 실험 중 | 정해진 근거 없음, 민감 |


In [ ]:
#@title 우리 데이터와 공통 함수 준비 (실행만 하면 됩니다)
import csv, math, random
import numpy as np
from collections import defaultdict
from src.gaip_simulation.clustering import dbscan_distinct_days, haversine_m, percentile_nearest_rank

CSV_PATH = ROOT + "/data/fixtures/gaip_visit_events.csv"
CORE_M, CAP_M = 500.0, 2000.0
COLORS = ["#1D9E75", "#378ADD", "#EF9F27", "#D4537E", "#7F77DD", "#639922", "#D85A30", "#0F6E56"]

with open(CSV_PATH, encoding="utf-8") as f:
    ALL_ROWS = list(csv.DictReader(f))
for r in ALL_ROWS:
    r["latitude"] = float(r["latitude"]); r["longitude"] = float(r["longitude"])

def load_driver(driver_id):
    rows = [r for r in ALL_ROWS if r["driver_id"] == driver_id]
    fit = [r for r in rows if r["period_role"] == "baseline"]
    ev = [r for r in rows if r["period_role"] != "baseline"]
    return fit, ev

def offsets(events, anchor):
    lat0, lon0 = anchor
    return [((e["longitude"] - lon0) * 111_320.0 * math.cos(math.radians(lat0)),
             (e["latitude"] - lat0) * 111_320.0) for e in events]

def zone_clusters(events, labels):
    groups = defaultdict(list)
    for e, l in zip(events, labels):
        if l >= 0:
            groups[l].append(e)
    out = []
    for l, evs in sorted(groups.items()):
        lat = sum(e["latitude"] for e in evs) / len(evs)
        lon = sum(e["longitude"] for e in evs) / len(evs)
        d = [haversine_m(e["latitude"], e["longitude"], lat, lon) for e in evs]
        p90 = percentile_nearest_rank(d, 0.90)
        out.append({"lat": lat, "lon": lon, "r": max(CORE_M, min(p90, CAP_M))})
    return out

def coverage(eval_events, clusters):
    if not clusters or not eval_events:
        return 0.0
    inz = sum(1 for e in eval_events
              if any(haversine_m(e["latitude"], e["longitude"], c["lat"], c["lon"]) <= c["r"]
                     for c in clusters))
    return inz / len(eval_events) * 100

def run_hdbscan(events, anchor, mcs, ms):
    from sklearn.cluster import HDBSCAN
    xy = offsets(events, anchor)
    h = HDBSCAN(min_cluster_size=mcs, min_samples=ms, allow_single_cluster=True, copy=True)
    return h.fit_predict(np.array(xy)).tolist()

FIT, EV = load_driver("gaip-123")
ANCHOR = (sum(e["latitude"] for e in FIT) / len(FIT), sum(e["longitude"] for e in FIT) / len(FIT))

def panel(ax, labels, title, fit=None, ev=None, anchor=None):
    fit = fit if fit is not None else FIT
    ev = ev if ev is not None else EV
    anchor = anchor if anchor is not None else ANCHOR
    clusters = zone_clusters(fit, labels)
    cov = coverage(ev, clusters)
    for (x, y), l in zip(offsets(fit, anchor), labels):
        if l < 0:
            ax.scatter(x, y, marker="x", c="#999999", s=42, zorder=3)
        else:
            ax.scatter(x, y, c=COLORS[l % len(COLORS)], s=30, zorder=3, edgecolors="none")
    for i, c in enumerate(clusters):
        cx = (c["lon"] - anchor[1]) * 111_320.0 * math.cos(math.radians(anchor[0]))
        cy = (c["lat"] - anchor[0]) * 111_320.0
        ax.add_patch(plt.Circle((cx, cy), c["r"], fill=True, alpha=0.10,
                                color=COLORS[i % len(COLORS)], zorder=1))
        ax.add_patch(plt.Circle((cx, cy), c["r"], fill=False, linestyle="--", linewidth=1.4,
                                color=COLORS[i % len(COLORS)], zorder=2))
    noise = sum(1 for l in labels if l < 0)
    ax.set_title(f"{title}\n거점 {len(clusters)}곳 · 노이즈 {noise}건 · 생활권이 담은 방문 {cov:.0f}%",
                 fontsize=11)
    ax.set_aspect("equal"); ax.grid(alpha=0.25)
    ax.set_xlabel("동쪽 (m)"); ax.set_ylabel("북쪽 (m)")
print("공통 코드 준비 완료")

---

# 2부. 우리 설계에 적용하면

## 2-1. 우리 파이프라인

어느 군집 방법을 쓰든 흐름은 같습니다.

```text
방문 좌표(기준선 2개월) → 뭉친 곳 찾기(군집화) → 거점마다 원 씌우기 → 원 합집합 = 생활권 → 이후 주행을 안/밖 판정
```

군집 방법이 바꾸는 건 "뭉친 곳 찾기" 한 단계뿐이고, 나머지(원, 합집합, 판정)는 공통입니다.
이 구조 때문에 뒤에서 보게 될 중요한 성질이 생깁니다. 군집이 다소 다르게 나와도
원을 씌워 합치는 순간 최종 생활권은 비슷해진다는 것.

## 2-2. 우리가 표준 DBSCAN에서 바꾼 것

작은 예제로 먼저 감을 잡겠습니다. 어떤 어르신의 2주 방문 12개:
집 6번(월~토), 시장 3번(화·목·토), 병원 2번(수·금), 결혼식장 1번(일).

우리 규칙은 한 문장입니다. "반경 260m 안에 서로 다른 3일 이상 방문이 모이면 거점."
적용하면 집(6일)과 시장(3일)은 거점이 되고, 병원(2일)과 결혼식장(1일)은 기준 미달로 노이즈가 됩니다.

In [ ]:
#@title 그림 — 미니 예제: 방문 12개에 규칙을 적용하면
import math as _m

MINI = [
    (10, 20, "월", "집"), (-30, -10, "화", "집"), (25, -35, "수", "집"),
    (-15, 40, "목", "집"), (40, 5, "금", "집"), (-40, -30, "토", "집"),
    (590, 140, "화", "시장"), (620, 165, "목", "시장"), (605, 130, "토", "시장"),
    (340, -445, "수", "병원"), (365, -460, "금", "병원"),
    (-650, 380, "일", "결혼식장"),
]
PLACE_COLOR = {"집": "#1D9E75", "시장": "#378ADD"}
ANCHORS = {"집": (0, 0), "시장": (605, 145), "병원": (352, -452), "결혼식장": (-650, 380)}

def draw_points(ax, color_by_place=True, noise_places=("병원", "결혼식장"), one_color=None):
    for (x, y, d, p) in MINI:
        if one_color:
            ax.scatter(x, y, c=one_color, s=45, zorder=3)
        elif color_by_place and p in PLACE_COLOR:
            ax.scatter(x, y, c=PLACE_COLOR[p], s=45, zorder=3)
        elif p in noise_places:
            ax.scatter(x, y, marker="x", c="#999999", s=60, zorder=3)
        else:
            ax.scatter(x, y, c="#888780", s=45, zorder=3)
    for name, (x, y) in ANCHORS.items():
        ax.annotate(name, (x, y), textcoords="offset points", xytext=(0, 26),
                    ha="center", fontsize=11)

fig, axes = plt.subplots(1, 3, figsize=(15, 5.2))
ax = axes[0]
draw_points(ax)
for name, r, c in [("집", 500, "#1D9E75"), ("시장", 500, "#378ADD")]:
    x, y = ANCHORS[name]
    ax.add_patch(plt.Circle((x, y), r, fill=True, alpha=0.08, color=c))
    ax.add_patch(plt.Circle((x, y), r, fill=False, linestyle="--", color=c))
ax.set_title("우리 규칙의 답 — 3일 채운 집·시장만 거점\n병원(2일)·결혼식장(1일)은 노이즈(×)", fontsize=11)

ax = axes[1]
draw_points(ax, color_by_place=True, noise_places=())
for name in ("집", "시장", "병원"):
    x, y = ANCHORS[name]
    ax.add_patch(plt.Circle((x, y), 90, fill=False, linestyle=":", color="#888780"))
ax.set_title("HDBSCAN이 보는 과정의 한 장면(반경 약 90m)\n각 장소가 먼저 각자 뭉친다", fontsize=11)

ax = axes[2]
draw_points(ax, one_color="#D4537E")
ax.add_patch(plt.Circle((60, -60), 780, fill=False, linestyle="--", color="#D4537E"))
ax.set_title("반경을 700m 이상 키우면 전부 한 덩어리\n어디서 멈추느냐가 곧 답이 된다", fontsize=11)

for ax in axes:
    ax.set_aspect("equal"); ax.grid(alpha=0.2)
    ax.set_xlim(-900, 900); ax.set_ylim(-750, 700)
fig.tight_layout(); plt.show()

표준과 다른 점을 정리하면 네 가지입니다.

| # | 표준 (sklearn) | 우리 구현 | 이유 |
|---|---|---|---|
| 1 | 유클리드 거리 (좌표 단위) | haversine 미터 거리 | 1부 실험 1에서 봤듯 eps는 축척에 민감. 위경도 도 단위는 방향에 따라 길이가 달라 왜곡됨 |
| 2 | min_samples = 이웃 점 개수 | 서로 다른 방문일 3일 이상 | 하루에 10번 정차한 곳(이삿날, 행사)이 거점으로 오인되는 걸 막음. 상품 규칙이 알고리즘 안에 들어간 형태 |
| 3 | 군집이 최종 출력 | 군집 위에 원 합집합 층 추가 | 판정의 관심사는 군집 모양이 아니라 "반복 방문 지역 안인가"라서 |
| 4 | 파라미터는 사용자 임의 | eps 값의 출처와 갱신 절차를 문서화 | 4부에서 정리 |

2번이 실제로 뭘 막는지 확인해 보겠습니다. 실제 운전자 데이터에 "하루에만 10번 정차한 가짜 지점"을
심어 놓고, 표준 DBSCAN과 우리 변형을 같은 조건으로 돌려봅니다.

In [ ]:
#@title 실험 — 하루 10번 정차한 지점의 운명
fit0, _ = load_driver("gaip-003")
anchor_lat = sum(e["latitude"] for e in fit0) / len(fit0)
anchor_lon = sum(e["longitude"] for e in fit0) / len(fit0)

rr = random.Random(3)
fake_lat = anchor_lat + 2000 / 111_320.0
one_day_burst = [{"latitude": fake_lat + rr.gauss(0, 25) / 111_320.0,
                  "longitude": anchor_lon + rr.gauss(0, 25) / 88_000.0,
                  "visit_date": "2025-11-15"} for _ in range(10)]
mixed = [{"latitude": e["latitude"], "longitude": e["longitude"], "visit_date": e["visit_date"]}
         for e in fit0] + one_day_burst

from sklearn.cluster import DBSCAN as SkDBSCAN
xy = np.array([((e["longitude"] - anchor_lon) * 111_320.0 * math.cos(math.radians(anchor_lat)),
                (e["latitude"] - anchor_lat) * 111_320.0) for e in mixed])
vanilla = SkDBSCAN(eps=260, min_samples=3).fit_predict(xy)
masil = dbscan_distinct_days(mixed, eps_m=260, min_distinct_days=3)["labels"]

def burst_result(labels):
    return "거점으로 인정" if max(labels[-10:]) >= 0 else "노이즈로 걸러냄"

print("하루 10번 정차한 가짜 지점의 운명")
print(f"  표준 DBSCAN(점 3개 기준)   -> {burst_result(list(vanilla))}")
print(f"  우리 변형(서로 다른 3일)    -> {burst_result(masil)}")

## 2-3. 우리 데이터에서 둘을 나란히

이제 실제 운전자 한 명(광역 지역, 거점 3곳을 오가는 사람)에게 네 가지 설정을 적용해 봅니다.
그림 읽는 법: 점 = 기준선 2개월 방문, 색 = 군집, 회색 × = 노이즈, 점선 원 = 거점의 생활권 원.
제목의 "담은 방문"은 이후 12개월 방문 중 생활권 안에 들어온 비율입니다.

In [ ]:
#@title 그림 — DBSCAN: eps를 다르게 하면 (같은 사람)
fig, axes = plt.subplots(1, 2, figsize=(13.5, 7.5))
panel(axes[0], dbscan_distinct_days(FIT, eps_m=260, min_distinct_days=3)["labels"],
      "eps=260m — 거점 3곳이 각자 잡힘")
panel(axes[1], dbscan_distinct_days(FIT, eps_m=1100, min_distinct_days=3)["labels"],
      "eps=1,100m — 남쪽 두 거점이 하나로 합쳐짐")
fig.tight_layout(); plt.show()

In [ ]:
#@title 그림 — HDBSCAN: 설정을 다르게 하면 (같은 사람)
fig, axes = plt.subplots(1, 2, figsize=(13.5, 7.5))
panel(axes[0], run_hdbscan(FIT, ANCHOR, 5, 3), "설정 5/3 — DBSCAN 260과 사실상 같은 결과")
panel(axes[1], run_hdbscan(FIT, ANCHOR, 3, 2), "설정 3/2 — 남쪽 거점이 여러 조각으로")
fig.tight_layout(); plt.show()

두 그림에서 보이는 것.

첫째, DBSCAN 260과 HDBSCAN 5/3은 사실상 같은 답을 냅니다. 우리 데이터의 평상시 산포(수십 m)가
eps보다 한참 작아서, 1부 실험 2의 "밀도 혼합" 상황이 발생하지 않기 때문입니다.

둘째, 각자의 흔들리는 방향이 다릅니다. DBSCAN은 eps를 크게 잡으면 거점이 합쳐지고(왼쪽 그림의 1,100m),
HDBSCAN은 설정을 느슨하게 하면 거점이 조각납니다(오른쪽 그림의 3/2).

셋째, 조각나도 "담은 방문"은 유지됩니다. 조각마다 원이 붙고 합집합은 비슷해지기 때문입니다.
그래서 조각남은 틀린 판정이 아니라 내부 표현의 차이입니다. 조각 수가 영향을 주는 곳은
판정이 아니라 생활권 지도 화면(거점을 링으로 그리는 부분)과 "생활권 N곳" 같은 서술 정도이고,
표시할 때만 가까운 조각을 합쳐 그리면 그마저 사라집니다.

개인정보 관련 참고. 우리는 거점이 병원인지 가족 집인지 알아내지 않습니다(POI 조회 안 함).
이게 가능한 이유가 위 구조입니다. 판정은 좌표·반복성·거리만 쓰고 장소의 의미를 쓰지 않으므로,
정체를 몰라도 판정 품질에 영향이 없습니다. 데모 화면에 보이는 장소 이름은 합성 시나리오의 이름이지
실제 조회 결과가 아닙니다.

## 2-4. 직접 바꿔보기

아래 셀 상단의 값 4개를 바꾸고 다시 실행하면(셀 클릭 후 Shift+Enter) 다른 운전자, 다른 설정으로
바로 비교해볼 수 있습니다. 먼저 대표 운전자 18명(유형 6 × 지역 3) 목록이 출력됩니다.

In [ ]:
DRIVER_ID = "gaip-123"   # 아래 목록에서 골라 바꿔보세요
EPS_M = 260              # DBSCAN: 한 장소로 묶는 반경(m). 100~1200 사이로 바꿔보세요
MIN_DAYS = 3             # DBSCAN: 거점 인정에 필요한 서로 다른 방문일
MCS, MS = 5, 3           # HDBSCAN 설정. 3,2로 바꾸면 조각나는 걸 볼 수 있습니다

reps = {}
for r in ALL_ROWS:
    reps.setdefault((r["environment_id"], r["designed_type"]), r["driver_id"])
print(f"{'지역':22s} {'유형':24s} 대표 운전자")
for (env, t), d in sorted(reps.items()):
    print(f"{env:22s} {t:24s} {d}")

fit_x, ev_x = load_driver(DRIVER_ID)
anchor_x = (sum(e["latitude"] for e in fit_x) / len(fit_x),
            sum(e["longitude"] for e in fit_x) / len(fit_x))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 7.5))
panel(axes[0], dbscan_distinct_days(fit_x, eps_m=EPS_M, min_distinct_days=MIN_DAYS)["labels"],
      f"DBSCAN eps={EPS_M}m + {MIN_DAYS}일 — {DRIVER_ID}", fit=fit_x, ev=ev_x, anchor=anchor_x)
panel(axes[1], run_hdbscan(fit_x, anchor_x, MCS, MS),
      f"HDBSCAN {MCS}/{MS} — {DRIVER_ID}", fit=fit_x, ev=ev_x, anchor=anchor_x)
fig.tight_layout(); plt.show()

---

# 3부. 둘 다 갖고 있는 이슈

어느 쪽이 정답이라는 얘기를 하려는 게 아닙니다. 각자 어떤 이슈가 있고,
그 이슈를 관리할 방법이 있는지 없는지를 보는 게 목적입니다.

## 먼저 결론부터: 두 방식의 남은 이슈는 사실상 같은 종류입니다

둘 다 계산은 잘 되고, 노이즈도 잘 걸러내고, 우리 데이터에서 결과도 동급입니다.
남은 이슈는 양쪽 모두 "**사람이 정하는 값이 있고, 그 값의 근거를 만들어야 한다**"는 것입니다.

- DBSCAN: eps(몇 미터)를 정해야 함
- HDBSCAN: min_cluster_size, min_samples(몇 개)를 정해야 함

HDBSCAN이 eps를 없앤 건 맞지만, 정해야 할 값 자체가 사라진 게 아니라 단위가
미터에서 개수로 바뀐 것입니다. 그래서 "왜 260이냐"가 "왜 5/3이냐"로 바뀔 뿐입니다.

완전히 대칭은 아니고, 차이가 두 개 있습니다.

1. **근거를 만드는 길.** eps는 미터라서 물리 세계를 잴 수 있습니다 — 주차 산포를 측정하면
   "재보니 이 값"이라고 답할 수 있습니다(4부의 k-거리). 개수 파라미터는 대응하는 물리량이
   없어서 민감도 실험("이 범위에서는 결과가 안 흔들린다")이 최선입니다. 참고로
   min_cluster_size는 "서로 다른 3일" 규칙에서 근거를 가져올 수 있어서(3일 조건을 후처리로
   걸면 됨), 순수하게 근거가 없는 건 min_samples 하나입니다.
2. **경계 사례의 이유 문장.** "왜 제 병원이 거점이 아닌가요?"에 DBSCAN은 "기준 기간에
   서로 다른 3일 미만이어서요"라고 답할 수 있고, HDBSCAN은 "내부 계산상 그렇습니다"가 됩니다.
   재현은 양쪽 다 되지만, 문장은 한쪽만 됩니다.

이 두 가지를 얼마나 무겁게 볼지는 판단의 문제입니다. 가볍게 본다면 HDBSCAN도 완전히
정당한 선택입니다.

## 각자의 이슈 상세

DBSCAN — eps를 잘못 정했을 때의 대가 (양방향):

| eps를 | 생기는 일 | 우리가 확인한 것 |
|---|---|---|
| 너무 크게 | 서로 다른 거점이 합쳐지고, 합쳐진 덩어리의 원이 상한(2km)에 걸려 생활권이 덜 덮임 | 2-3의 1,100m 그림. 담은 방문 100 → 96% |
| 너무 작게 (산포보다 작게) | 산포가 큰 동네에서 거점이 조각나고 방문이 버려짐 | 산포를 400m로 키운 실험에서 방문 20%를 버리고 생활권이 아예 안 만들어진 사람도 나옴 (부록 A) |

HDBSCAN — 설정을 잘못 정했을 때의 대가:

| 설정을 | 생기는 일 | 우리가 확인한 것 |
|---|---|---|
| 느슨하게 (3/2) | 거점이 여러 조각으로 나뉘고 노이즈가 늘어남. 판정은 유지되지만 지도 표시가 지저분해짐 | 2-3 그림. 우리 180명 실측에서 조각 비율 47% |
| 엄격하게 (5/3) | DBSCAN과 사실상 같은 결과 | 조각 비율 12%, 담은 방문 동일 |

참고로 "기준을 미리 문서에 고정할 수 없다"는 건 HDBSCAN의 이슈가 아닙니다.
미리 고정해야 하는 건 숫자가 아니라 정하는 방식이고(개인별 P90 반경이 이미 그 구조 —
방식만 고정, 값은 고객마다 산출), HDBSCAN도 "이 설정을 적용한다"는 방식 고정은 됩니다.

## 공통 이슈 (군집 방법과 무관한 것)

| # | 이슈 | 확인한 것 |
|---|---|---|
| 1 | 원 반경 상한(2km)이 아주 넓은 생활권을 다 못 덮음 | 광역 지역 커버리지가 다른 지역보다 낮음 (86% vs 91%). 알고리즘이 아니라 원 규칙의 문제 |
| 2 | 최소 반경 500m가 커서 새 목적지 방문을 기존 생활권이 흡수 | 새 목적지 방문의 62%가 기존 존 안으로 들어옴. 어느 쪽을 써도 동일 |
| 3 | 합성 데이터 검증의 한계 | 여기 나온 모든 수치는 우리가 만든 데이터 기준. 실데이터로 재기 전에는 어느 쪽도 검증됐다고 말할 수 없음 |


---

# 4부. eps 값의 근거 — 조사한 것들 정리

"왜 260m냐"에 대해 조사했던 내용입니다. 요약하면, 절대값을 주는 문헌은 없고,
공간 임계의 대역을 주는 문헌들과 방문일 기준의 선례가 있습니다.

| 확인한 것 | 내용 | 출처 |
|---|---|---|
| 장소 판정 반경의 대역 | GPS 궤적 연구가 체류지 판정에 200m 사용 | Zheng et al., WWW 2009 (Microsoft GeoLife) |
| 〃 | 스마트폰 데이터 관심장소 연구가 250m 사용, 200~300m 권장 | Montoliu et al., 2013 |
| "서로 다른 3일"의 선례 | 통신 데이터로 중요 장소를 찾은 연구가 60일 관찰에서 "전체 일수의 5% = 3일"을 기준으로 사용. 횟수가 아니라 방문한 날 수로 세는 이유까지 우리와 동일 | Isaacman et al., Pervasive 2011 — [원문 PDF](https://mrmgroup.cs.princeton.edu/papers/Isaacman_pervasive11.pdf) |
| 고정 eps의 한계는 원저자도 인정 | "전역 eps는 밀도가 다른 군집을 병합할 수 있다" (p.229) | Ester et al., KDD 1996 — [원문 PDF](http://cdn.aaai.org/KDD/1996/KDD96-037.pdf) |
| 지역별로 다른 임계가 필요하다는 실증 | 도시권 클러스터링에서 나라별 최적 eps가 120m~1,000m로 크게 다르고, 균일 eps는 실패했다고 보고 | MDPI Applied Sciences 15(22):12278 (집계 데이터 대상이라 개인 생활권과 스케일이 다름. 참고 수준으로만) |

현재 값의 논리: 도심 260m는 문헌 대역(200~300m)의 가운데. 교외 520, 광역 1,100은
260에 2배씩 곱한 축척 규칙입니다(환경 설계 전체와 같은 비율). 즉 임의 숫자 3개가 아니라
"문헌 대역 기준값 하나 + 일관된 규칙 하나"인데, 뒤의 2배 규칙 자체는 검증된 게 아니라 설계 가정입니다.

푸아송 오류율 계산, 국내 방문빈도 통계(건강보험, 노인실태조사) 등 전체 근거는 저장소의
[final 폴더 근거 노트](https://github.com/summit1123/seniorcareservice/tree/claude/gaip-dashboard-refine/final)에 원문 인용으로 정리돼 있습니다.

## 운영에서는 값을 "정하지 않고 재는" 방향

미리 문서에 적어야 하는 건 숫자가 아니라 정하는 방식이라고 했습니다. eps도 상수 대신
측정 절차를 고정하는 방향입니다. 그 절차가 k-거리입니다.

k-거리 = 어떤 방문점에서, 다른 날의 방문점 중 k번째로 가까운 것까지의 거리.

| 기준 | 값 | 이유 |
|---|---|---|
| k | 2 | 거점 규칙이 "서로 다른 3일"이므로, 거점 후보가 되려면 다른 날 이웃이 2개 필요. 그래서 "2번째 다른 날 이웃까지 거리"가 같은 장소 재방문의 흩어짐 폭이 됨 |
| 다른 날만 셈 | - | 같은 날 여러 정차는 재방문이 아니라 한 외출이므로 |
| 컷 | P95 | 재방문 쌍의 95%를 같은 장소로 인정하는 반경. 꼬리 5%는 이상치로 배제. 이 백분위가 사람이 정하는 마지막 값이라는 건 남는 사실 |

아래는 이 절차를 합성 데이터에 돌려본 것입니다. 나오는 숫자는 합성 산포일 뿐이라 260의 근거가
아니고(우리가 만든 데이터로는 정당화도 반박도 안 됩니다), 지역별로 산포가 다르면 절차가 그 차이를
잡아낸다는 것만 확인하면 됩니다. 진짜 데이터에서 작동하는지는 부록 B에서 봅니다.

In [ ]:
#@title 표 — k-거리 측정을 합성 데이터에 돌려보면
by_env_driver = defaultdict(lambda: defaultdict(list))
for r in ALL_ROWS:
    if r["period_role"] == "baseline":
        by_env_driver[r["environment_id"]][r["driver_id"]].append(
            (r["latitude"], r["longitude"], r["visit_date"]))

def pct(xs, q):
    xs = sorted(xs)
    return xs[max(0, min(len(xs) - 1, int(len(xs) * q)))]

ENV_KO = {"dense_urban": "도심", "suburban_mid_density": "교외", "wide_low_density": "광역"}
print(f"{'지역':8s} {'중앙값':>7s} {'P90':>7s} {'P95':>7s}")
for env, drivers in by_env_driver.items():
    pooled = []
    for d, pts in drivers.items():
        for i, (la, lo, dt) in enumerate(pts):
            ds = sorted(haversine_m(la, lo, la2, lo2)
                        for j, (la2, lo2, dt2) in enumerate(pts) if j != i and dt2 != dt)
            if len(ds) >= 2:
                pooled.append(ds[1])
    print(f"{ENV_KO[env]:8s} {pct(pooled,0.50):6.0f}m {pct(pooled,0.90):6.0f}m {pct(pooled,0.95):6.0f}m")
print()
print("합성 산포(수십 m)는 실제(GPS 오차 + 주차장 크기)보다 훨씬 작습니다. 값이 아니라 절차를 보는 표입니다.")

---

# 5부. 지금 고민하고 있는 것들

아직 결정된 것이 없는 질문들입니다. 아래에 "제안"이라고 쓴 것은 이 자료를 만드는 과정에서
나온 생각이지 팀에서 정한 게 아닙니다. 회의에서 같이 정할 부분이라 질문 그대로 적습니다.

**1. 지역 3단(260/520/1,100)을 유지할까, 단일값으로 갈까.**
합성 실험에서는 큰 지역값이 오히려 손해였습니다(2-3의 병합 그림). 그런데 합성 산포가 실제보다
촘촘해서 이 결과로 확정할 수도 없습니다. 제안: 지역 구분 유지 여부를 미리 정하지 말고,
실데이터 k-거리를 지역별로 재서 지역 간 차이가 크면 지역별 값, 작으면 단일값.

**2. HDBSCAN으로 갈 이유가 생기는 조건은.**
성능은 우리 데이터에서 동급입니다. 남은 차이는 경계 사례에서 이유를 문장으로 줄 수 있느냐(DBSCAN 쪽)와
정해야 할 값의 근거를 측정으로 만들 수 있느냐(역시 DBSCAN 쪽이 길이 곧음), 반대로 eps 논쟁 자체가
없어진다는 것(HDBSCAN 쪽)입니다. 산포 실험(부록 A)에서는 산포가 eps를 넘어서는 구간에서 HDBSCAN이
더 잘 버텼습니다. 제안: 실데이터에서 산포가 eps급으로 큰 지역이 흔하게 확인되면 그 지역부터
eps 상향 또는 HDBSCAN 전환을 검토. 현행이 DBSCAN 변형인 것은 성능 우위가 아니라, 민원·감사에서의
문장 가능성과 이미 구현·검증이 끝나 있다는 실무 사정 때문입니다.

**3. 원 규칙(최소 500m, 상한 2km) 자체의 손질.**
광역 커버리지 86%와 새 목적지 흡수 62%는 군집 방법이 아니라 이 층의 숙제입니다.
알고리즘 논쟁과 분리해서 다뤄야 합니다.

**4. 합성 데이터의 한계.**
이 노트북의 모든 수치는 우리가 설계한 데이터에서 나온 것입니다. 파이프라인이 돌아가고
절차가 작동한다는 확인까지는 되지만, 성능이 검증됐다는 뜻은 아닙니다. 실데이터 파일럿에서
정답 라벨 없이도 계산되는 지표(커버리지, 노이즈, 거점 수, 기간 안정성)로 다시 재는 게 다음 단계입니다.

---

# 부록

## 부록 A — 산포를 키우면 언제 갈라지나

주차 산포를 인위적으로 키우면서 광역 60명을 다시 돌린 실험입니다. 산포가 eps(260m)에
가까워지고 넘어서는 순간부터 두 방법이 갈라집니다. DBSCAN은 조각나고 방문을 버리기 시작하고,
HDBSCAN은 반경을 늘려 버팁니다. "차이가 나는 조건"을 보는 실험입니다.

In [ ]:
#@title 실험 — 산포 0/200/400m에서의 비교
def jitter(events, sigma, rng):
    out = []
    for e in events:
        out.append({"latitude": e["latitude"] + rng.gauss(0, sigma) / 111_320.0,
                    "longitude": e["longitude"] + rng.gauss(0, sigma)
                    / (111_320.0 * math.cos(math.radians(e["latitude"]))),
                    "visit_date": e["visit_date"]})
    return out

wide = defaultdict(lambda: {"fit": [], "ev": []})
for r in ALL_ROWS:
    if r["environment_id"] != "wide_low_density":
        continue
    e = {"latitude": r["latitude"], "longitude": r["longitude"], "visit_date": r["visit_date"]}
    wide[r["driver_id"]]["fit" if r["period_role"] == "baseline" else "ev"].append(e)

print(f"{'산포':>6s} | {'방식':16s} | {'생활권 미형성':>8s} | {'평균 거점':>6s} | {'버려진 방문':>7s} | {'담은 방문':>7s}")
for sigma in [0, 200, 400]:
    for name, fn in [("DBSCAN 260m/3일",
                      lambda f: dbscan_distinct_days(f, eps_m=260, min_distinct_days=3)["labels"]),
                     ("HDBSCAN 5/3",
                      lambda f: run_hdbscan(f, (sum(e['latitude'] for e in f) / len(f),
                                                sum(e['longitude'] for e in f) / len(f)), 5, 3))]:
        rng2 = random.Random(42)
        zero = 0; hubs = []; noise = 0; pts = 0; covs = []
        for d, dd in wide.items():
            fitj = jitter(dd["fit"], sigma, rng2)
            evj = jitter(dd["ev"], sigma, rng2)
            labels = fn(fitj)
            zs = zone_clusters(fitj, labels)
            if not zs:
                zero += 1
            hubs.append(len(zs)); noise += sum(1 for l in labels if l < 0); pts += len(labels)
            covs.append(coverage(evj, zs))
        print(f"{sigma:5d}m | {name:16s} | {zero:6d}명 | {sum(hubs)/len(hubs):8.1f} "
              f"| {noise/pts*100:6.1f}% | {sum(covs)/len(covs):6.1f}%")

## 부록 B — 진짜 GPS 데이터로 확인

본문 수치는 전부 합성 데이터 기준이라, 우리가 만들지 않은 데이터에서도 규칙이 상식적으로
작동하는지 한 번은 봐야 합니다. 공개된 실제 GPS를 씁니다. Geoff Boeing이라는 연구자가 공개한
2014년 여름 유럽 여행 기록 1,759개(위경도와 날짜 포함)입니다.

"서로 다른 3일 이상 머문 곳 = 거점" 규칙을 그대로 적용하면, 여행자가 실제로 오래 머문 도시들이
이름으로 나옵니다. 결과가 상식과 맞는지 직접 확인할 수 있습니다.

여기서는 eps를 500m로 씁니다. 이 데이터의 "장소"는 주차장이 아니라 도시 안 체류지(숙소 일대)라
물리적 크기가 달라서입니다. eps는 그 데이터에서 한 장소의 크기에 맞춘다는 원칙의 예입니다.

더 큰 실데이터가 필요하면: Microsoft GeoLife (베이징 182명 GPS, https://www.microsoft.com/en-us/download/details.aspx?id=52367),
SNAP Brightkite 체크인 (https://snap.stanford.edu/data/loc-Brightkite.html).

In [ ]:
#@title 실험 — 실제 여행 GPS 1,759개에 우리 규칙 적용
import urllib.request, io, csv as _csv
from collections import Counter

URL = "https://raw.githubusercontent.com/gboeing/2014-summer-travels/master/data/summer-travel-gps-full.csv"
raw = urllib.request.urlopen(URL).read().decode()
rows = list(_csv.DictReader(io.StringIO(raw)))
real = [{"latitude": float(r["lat"]), "longitude": float(r["lon"]),
         "visit_date": r["date"].split()[0], "city": r["city"]} for r in rows]
print(f"실데이터 로드: GPS {len(real)}개 (2014-05 ~ 2014-08, 유럽)")

res = dbscan_distinct_days(real, eps_m=500, min_distinct_days=3)
labels_real = res["labels"]

groups = defaultdict(list)
for e, l in zip(real, labels_real):
    if l >= 0:
        groups[l].append(e)
hubs = sorted(groups.values(), key=lambda g: -len({e["visit_date"] for e in g}))
print(f"\n발견된 거점 {len(hubs)}곳 / 노이즈 {res['noise_count']}개 (일회성 이동 경로)")
print(f"\n{'거점(최빈 도시)':24s} {'머문 날':>6s} {'GPS 수':>6s}")
for g in hubs[:8]:
    city = Counter(e["city"] for e in g).most_common(1)[0][0]
    print(f"{city:24s} {len({e['visit_date'] for e in g}):5d}일 {len(g):6d}개")

fig, ax = plt.subplots(figsize=(9, 7))
for e, l in zip(real, labels_real):
    if l < 0:
        ax.scatter(e["longitude"], e["latitude"], marker="x", c="#bbbbbb", s=14, zorder=2)
    else:
        ax.scatter(e["longitude"], e["latitude"], c=COLORS[l % len(COLORS)], s=22, zorder=3)
for g in hubs[:6]:
    city = Counter(e["city"] for e in g).most_common(1)[0][0]
    la = sum(e["latitude"] for e in g) / len(g)
    lo = sum(e["longitude"] for e in g) / len(g)
    ax.annotate(city, (lo, la), textcoords="offset points", xytext=(6, 6), fontsize=10)
ax.set_title("실제 여행 GPS — 색 = 서로 다른 3일 이상 머문 거점, 회색 × = 일회성 경로", fontsize=11)
ax.set_xlabel("경도"); ax.set_ylabel("위도"); ax.grid(alpha=0.25)
fig.tight_layout(); plt.show()